# MSTAR Classical Machine Learning Benchmarks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohitgit1/Target-Detection-in-MSTAR-Images/blob/main/notebooks/03_classical_ml_benchmarks.ipynb)

Modernized implementation of `Project_21C02.ipynb`. Replaces deprecated `scipy.misc` with standard Pillow/NumPy, and benchmarks **PCA + SVM (RBF)**, **Random Forest**, **Gradient Boosted Decision Trees**, and **MLP** on MSTAR.

In [ ]:
import sys
!pip install -q scikit-learn matplotlib pillow
sys.path.append('..')

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from mstar_atr.constants import CLASSES, CLASS_TO_IDX
from mstar_atr.data.downloader import prepare_mstar_dataset
from mstar_atr.models.classical import ClassicalSARPipeline

## 1. Load Data as Arrays

In [ ]:
data_dir = prepare_mstar_dataset('../data/mstar', verbose=False)

def load_dataset_arrays(split_dir):
    X, y = [], []
    for cls_name in CLASSES:
        cls_folder = os.path.join(split_dir, cls_name)
        if not os.path.isdir(cls_folder):
            continue
        for fname in os.listdir(cls_folder):
            if fname.lower().endswith(('.jpeg', '.jpg', '.png')):
                p = os.path.join(cls_folder, fname)
                with Image.open(p) as img:
                    arr = np.array(img.convert('L'), dtype=np.float32)
                    X.append(arr)
                    y.append(CLASS_TO_IDX[cls_name])
    return np.array(X), np.array(y)

X_train, y_train = load_dataset_arrays(os.path.join(data_dir, 'train'))
X_test, y_test = load_dataset_arrays(os.path.join(data_dir, 'test'))
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

## 2. Benchmark Classical Classifiers

In [ ]:
classifiers = ['svm', 'rf', 'gbdt', 'mlp', 'dt', 'knn']
benchmark_results = {}

for clf_name in classifiers:
    pipe = ClassicalSARPipeline(classifier_type=clf_name, n_pca_components=30)
    pipe.fit(X_train, y_train)
    train_score = pipe.score(X_train, y_train)
    test_score = pipe.score(X_test, y_test)
    benchmark_results[clf_name.upper()] = (train_score, test_score)
    print(f'{clf_name.upper():<6} | Train Acc: {train_score*100:.1f}% | Test Acc: {test_score*100:.1f}%')

## 3. Visualize Comparison

In [ ]:
names = list(benchmark_results.keys())
test_accs = [benchmark_results[k][1] * 100 for k in names]

plt.figure(figsize=(9, 4))
bars = plt.bar(names, test_accs, color='#0ea5e9')
plt.title('Classical ML Performance on MSTAR (PCA-30)')
plt.ylabel('Test Accuracy (%)')
plt.ylim(0, 105)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 1.5, f'{yval:.1f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()